In [4]:
!git clone https://github.com/GyanRout/Major-Projects.git
%cd Major-Projects
!git pull

Cloning into 'Major-Projects'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 64 (delta 16), reused 44 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 1.70 MiB | 13.09 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/Major-Projects/Major-Projects
Already up to date.


In [5]:
!pip install -r /content/Major-Projects/dp_recommender_project/requirements.txt

In [6]:
import sys

# Adding project root directory to Python's path
project_root = '/content/Major-Projects/dp_recommender_project'
if project_root not in sys.path:
    sys.path.append(project_root)

In [7]:
from src.model_defs import RecommenderModel
from src.data_utils import Loader
from torch.utils.data import DataLoader
from sklearn.metrics import mean_squared_error
import numpy as np
import json
import pandas as pd
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

user_idx_path = '/content/Major-Projects/dp_recommender_project/data/user_to_index.json'
movie_idx_path = '/content/Major-Projects/dp_recommender_project/data/movie_to_index.json'
test_data_path = '/content/Major-Projects/dp_recommender_project/data/test_data.csv'
load_path = '/content/Major-Projects/dp_recommender_project/src/private_recommender.pth'

with open(user_idx_path,'r') as f:
  NUM_USER = len(json.load(f))

with open(movie_idx_path,'r') as f:
  NUM_ITEM = len(json.load(f))

test_df = pd.read_csv(test_data_path)
test_dataset = Loader(test_df)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

model = RecommenderModel(num_user=NUM_USER, num_item=NUM_ITEM)

loaded_state_dict = torch.load(load_path, map_location=device, weights_only=True)
model.load_state_dict(loaded_state_dict)
model = model.to(device)

In [9]:
def evalute_rmse(model, dataloader, device):
  model.eval()

  all_prediction=[]
  all_target=[]

  with torch.inference_mode():
    for users, items, rating in dataloader:
      users = users.to(device)
      items = items.to(device)

      pred = model(users, items)

      all_prediction.extend(pred.squeeze().cpu().numpy())
      all_target.extend(rating.numpy())

  mse = mean_squared_error(all_target, all_prediction)
  rmse = np.sqrt(mse)

  return rmse

RMSE = evalute_rmse(model, test_loader, device)

print(f"The RMSE of our private model is {RMSE:.4f}")

The RMSE of our private model is 0.9564
